# Context-Free Grammars

**Domain:** Procedural Generation  ·  **from study list**  ·  **runnable:** yes

A refresher on **context-free grammars (CFGs)** as a procedural-generation tool: write a set of
rewrite rules, start from a single symbol, and repeatedly expand non-terminals until only
literal text remains. The structure-aware cousin of the Markov chain — it gives you *nesting*,
*recursion*, and *guaranteed-valid* output that statistical models can't promise. The engine
behind quest/dialogue generators (Tracery), name forges, and any "fill-in-the-template" system.

## 1. What & Why

A **context-free grammar** is a set of **production rules** that rewrite **non-terminal** symbols
into sequences of other symbols, until you're left with only **terminals** (literal output). It's
"context-free" because each rule fires based only on the symbol being expanded — *never* on what
surrounds it. Formally a CFG is the 4-tuple `(N, Σ, R, S)`: non-terminals `N`, terminals `Σ`,
rules `R`, and a start symbol `S`.

For procedural generation the loop is:

1. **Author** rules — e.g. `greeting → "hail" | "well met" | "greetings"`.
2. **Expand** from the start symbol, picking a random production for each non-terminal you hit,
   recursing until everything is a terminal.

**The problem it solves.** You want output with *guaranteed structure*: a quest line that always
has a giver, an objective, and a reward; an expression with matched parentheses; a sentence that
is always grammatical. A Markov chain learns local statistics but can't promise any of that — it
has no notion of "a noun phrase" or "balanced brackets." A grammar encodes that scaffolding
directly, and every derivation is valid by construction.

**When to reach for it.** Template-driven text (dialogue, item descriptions, quests), name/word
generators with morphological structure, any output that must obey a syntax (math expressions,
config files, music phrases, building floor-plans), and mixed systems where a grammar supplies
the skeleton and another method fills the gaps.

**When not to.** When you have *examples* but no rules and want output that mimics their
statistics — reach for a Markov chain instead ([[markov-chains]]). When neighbors must satisfy
hard 2D/3D constraints, use Wave Function Collapse ([[wave-function-collapse]]). And CFGs can't
natively count or enforce agreement across distant symbols ("exactly N items," "this pronoun
must match that earlier noun") — that needs a context-sensitive or attributed grammar.

## 2. Mental Model

Think of **Mad Libs that can nest inside themselves**. You start with one blank labeled `STORY`.
Each blank has a little menu of ways to fill it, and a filling can introduce *more* blanks. You
keep resolving blanks at random until none are left — the page you're holding is your output, and
the record of *which choices you made* is a **parse tree**.

```
                 S  (start)
                 |
              quest
         /      |        \
      giver  objective   reward
        |       |   \        |
    "the king" verb  thing  "gold"
        |       |      |
     (term.)  "slay" "dragon"
```

Read **top-down** it's *generation*: expand `S` → `quest` → `giver objective reward` → … → text.
Read **bottom-up** it's *parsing*: given text, find the tree that produces it. Same grammar, both
directions. The key superpower over a Markov chain: because `objective` can expand into something
that *contains another* `objective`, the grammar can produce **arbitrarily deep nesting** —
matched parentheses, sub-clauses, fractal structure — which a flat state machine fundamentally
cannot.

## 3. Key Concepts

- **Terminal** — a literal symbol that appears in the final output and is never rewritten
  (`"dragon"`, `"+"`, `"a"`). The alphabet `Σ`.
- **Non-terminal** — a placeholder that must be expanded by a rule (`STORY`, `noun`, `expr`). The
  set `N`. Conventionally written in caps or angle-brackets `<expr>`.
- **Production rule** — `A → α`, meaning "`A` may be replaced by the symbol sequence `α`." The
  `|` is shorthand for alternative right-hand sides of the same non-terminal.
- **Start symbol `S`** — the single non-terminal you begin every derivation from.
- **Derivation** — the sequence of rule applications turning `S` into a terminal string. Choosing
  *which* alternative at each step is what makes generation procedural/random.
- **Parse tree (derivation tree)** — the tree of choices; its leaves read left-to-right are the
  output. Captures the *structure*, which a flat string loses.
- **Recursion** — a non-terminal that can (directly or indirectly) expand to itself. This is the
  source of unbounded nesting and infinite variety — and of runaway expansion if unguarded.
- **Ambiguity** — a grammar is ambiguous if some string has more than one parse tree. Fine for
  *generation*; a headache for *parsing* (e.g. the classic `expr + expr * expr` precedence trap).
- **Weighted / stochastic CFG** — attach probabilities (or counts) to each alternative so some
  expansions are more likely than others. This is how you tune the "feel" of the output.
- **BNF / EBNF** — the standard textual notations for writing grammars (`<x> ::= ...`); EBNF adds
  `?` `*` `+` repetition operators. Tracery is essentially a JSON-flavored stochastic CFG.

## 4. Setup

Pure standard-library Python — a CFG is just a dictionary mapping non-terminals to lists of
right-hand sides, plus `random` for choosing among them. Nothing to install; everything below
runs on CPU in microseconds.

For real projects, the [`tracery`](https://pypi.org/project/tracery/) package gives you the same
model with `#symbol#` template syntax and modifiers (`.capitalize`, `.a`, `.s`); we reimplement
its core in a few lines below so the notebook stays dependency-free and self-explanatory.

In [ ]:
# A CFG needs nothing beyond the standard library. The real `tracery` package is optional —
# install it only if you want its #symbol# syntax and modifiers:  %pip install -q tracery
import random
import re

rng = random.Random(7)  # seeded for reproducible output
print("stdlib only — ready")

## 5. Worked Examples

Two self-contained examples:

1. **A Tracery-style quest/flavor generator** — a *weighted* grammar over template strings.
   Shows the everyday procedural-text use: structured variety from a handful of rules.
2. **A recursive arithmetic-expression grammar** — generates *always-valid, arbitrarily nested*
   expressions and prints the derivation tree. This is the thing a Markov chain cannot do:
   guaranteed balanced structure via recursion.

### Example 1 — A weighted grammar for quest flavor text

Rules map a non-terminal to a list of `(weight, template)` alternatives. A template is a string
where `#name#` marks a non-terminal to expand. We expand the start symbol depth-first, picking
alternatives by weight, until no `#...#` markers remain. This is exactly Tracery's model.

In [ ]:
# A weighted context-free grammar. Each rule: non-terminal -> list of (weight, template).
grammar = {
    "quest": [(1, "#giver# needs a hero to #objective#. #reward#")],
    "giver": [
        (3, "The aging king"),
        (2, "A frightened villager"),
        (1, "The guild of #profession#s"),
    ],
    "profession": [(1, "blacksmith"), (1, "alchemist"), (1, "cartographer")],
    "objective": [
        (2, "#verb# the #adjective# #monster# of #place#"),
        (1, "recover the #adjective# #artifact# from #place#"),
    ],
    "verb": [(1, "slay"), (1, "banish"), (1, "outwit")],
    "adjective": [(1, "ancient"), (1, "cursed"), (1, "ravenous"), (1, "shadow-touched")],
    "monster": [(1, "dragon"), (1, "lich"), (1, "basilisk")],
    "artifact": [(1, "crown"), (1, "chalice"), (1, "tome")],
    "place": [(1, "the Sunken Vault"), (1, "Mount Cinder"), (1, "the Whispering Fen")],
    "reward": [
        (2, "The reward: #treasure#."),
        (1, "Succeed, and #treasure# is yours."),
    ],
    "treasure": [(1, "a chest of gold"), (1, "a knight's title"), (1, "a wish")],
}

TOKEN = re.compile(r"#(\w+)#")

def expand(symbol, grammar, depth=0):
    """Pick a weighted alternative for `symbol`, then recursively expand its markers."""
    if depth > 50:                       # guard against accidental infinite recursion
        return symbol
    alts = grammar[symbol]
    weights = [w for w, _ in alts]
    template = rng.choices(alts, weights=weights)[0][1]
    # Replace every #marker# in the chosen template by expanding it in turn.
    return TOKEN.sub(lambda m: expand(m.group(1), grammar, depth + 1), template)

for i in range(5):
    print(f"{i+1}. {expand('quest', grammar)}")

Five rules produce thousands of distinct quests, every one grammatically well-formed. The
weights bias the output (the king appears more often than the guild) without you enumerating
combinations. Note `giver → "The guild of #profession#s"` shows a non-terminal nested *inside* a
template — the structural composition Markov chains lack.

### Example 2 — Recursive grammar: always-valid arithmetic expressions

Here's the payoff of context-freeness. The grammar

```
expr   → term (('+' | '-') term)*          # via recursion below
expr   → expr OP expr  |  term
term   → '(' expr ')'  |  NUMBER
```

is **directly self-referential**: `expr` can contain `term`, which can contain `( expr )`. That
recursion is what lets it emit *arbitrarily deep, always-balanced* parentheses — something no
finite-state / Markov model can guarantee. We build the **parse tree** explicitly, then flatten
it to a string, and `eval` it to prove every generated expression is syntactically valid.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Node:
    symbol: str
    children: list = field(default_factory=list)  # Nodes or terminal strings

def gen_expr(depth=0):
    # Bias toward terminating as depth grows, so recursion is finite.
    stop = depth >= 3 or rng.random() < 0.4
    if stop:
        return Node("expr", [gen_term(depth)])
    op = rng.choice(["+", "-", "*"])
    return Node("expr", [gen_expr(depth + 1), op, gen_expr(depth + 1)])

def gen_term(depth=0):
    if depth < 3 and rng.random() < 0.35:
        return Node("term", ["(", gen_expr(depth + 1), ")"])  # parenthesized sub-expression
    return Node("term", [str(rng.randint(1, 9))])             # a NUMBER terminal

def flatten(node):
    if isinstance(node, str):
        return node
    return " ".join(flatten(c) for c in node.children)

rng.seed(42)
for _ in range(6):
    tree = gen_expr()
    text = flatten(tree)
    print(f"{text:<28} = {eval(text)}")   # eval proves it is valid, balanced arithmetic

In [ ]:
# Inspect the structure: print one derivation/parse tree.
def show(node, indent=0):
    pad = "  " * indent
    if isinstance(node, str):
        print(f"{pad}· {node!r}")          # terminal leaf
    else:
        print(f"{pad}{node.symbol}")
        for c in node.children:
            show(c, indent + 1)

rng.seed(5)
tree = gen_expr()
print("flattened:", flatten(tree), "=", eval(flatten(tree)))
print("\nparse tree:")
show(tree)

Every expression `eval`s without error because the grammar can only ever produce balanced,
well-formed output — validity is *structural*, not statistical. The parse tree is the explicit
record of the derivation; swap `eval`/`flatten` for a renderer and the same machinery emits
dungeon layouts, music phrases, or UI trees.

## 6. Gotchas & Pitfalls

- **Unbounded recursion blows the stack.** A rule like `expr → expr '+' expr` with no bias toward
  terminating will recurse forever (or explode combinatorially). Always add a depth cap, a
  termination probability that rises with depth, or a grammar designed to shrink — as both
  examples above do.
- **Left recursion is fine for *generation*, fatal for naive *parsing*.** `A → A x` loops a
  recursive-descent parser instantly. If you'll parse with your grammar, refactor left recursion
  away; if you only generate, it's just another recursion to bound.
- **Ambiguity bites parsers, not generators.** `expr → expr OP expr` happily *generates* but is
  ambiguous (no precedence), so parsing it is underspecified. Encode precedence with layered
  non-terminals (`expr` → `term` → `factor`) when the parse must be deterministic.
- **CFGs can't count or enforce agreement.** "Exactly 3 items," "the verb must agree with the
  subject chosen earlier," "close the same tag you opened" — these need context. Add attributes/
  parameters to the grammar (an *attributed* grammar) or post-process; don't expect plain CFG
  rules to remember earlier choices.
- **Forgetting to expand every marker.** If a template references `#widget#` but the grammar has
  no `widget` rule, you get a `KeyError` (or, worse, literal `#widget#` in the output). Validate
  that every referenced non-terminal has a rule.
- **Flat weights make output feel samey or absurd.** Uniform alternatives over-produce rare
  combinations. Tune weights (or use a stochastic CFG learned from data) so common cases dominate
  and edge cases stay rare.
- **No global coherence between branches.** Two sibling expansions don't know about each other, so
  you can generate "the *dead* hero will *celebrate*." Constraints across branches need attributes,
  a second pass, or a different model.
- **Reproducibility.** Generation is stochastic — seed the RNG (`random.Random(seed)`) when you
  need deterministic output for tests or save-files.

## 7. When to Use vs Alternatives

**Reach for a CFG when** your output has *structure you can name* — parts, nesting, a syntax — and
you can write the rules. Quests, dialogue, item/spell descriptions, structured names, expressions,
config/markup, music phrases, building layouts. You get guaranteed-valid output and big variety
from a tiny, editable rule set, with zero training data.

| Approach | Strength | Weakness vs. CFG | Use when |
|---|---|---|---|
| **Context-free grammar** | Guaranteed structure & nesting; tiny editable rules; no data needed | You must author rules; no statistics from examples; can't count/agree across branches | Output must obey a named structure or syntax |
| **Markov chain** ([[markov-chains]]) | Learns local style from examples; trivial to train | No nesting, no validity guarantees, no long-range structure | You have examples and want statistical mimicry |
| **Wave Function Collapse** ([[wave-function-collapse]]) | Hard local constraints over 2D/3D tiles | Not for linear/text structure; heavier; can backtrack | Tile maps where neighbors must be compatible |
| **L-systems** ([[l-systems]]) | Parallel rewriting; great for fractal/organic growth | Less natural for choice-driven branching text | Plants, fractals, recursive geometry |
| **Attributed / context-sensitive grammar** | Can enforce counting, agreement, references | More complex; loses pure context-freeness | Constraints that plain CFGs can't express |
| **Neural LM (Transformer)** | Long-range coherence, semantics, open-ended | Needs data, training, GPU; no hard guarantees | Coherent prose/code where meaning matters |

Rule of thumb: **grammar for structure, Markov for vibe, neural for meaning.** They compose well —
a grammar supplies the skeleton (`#name#`, `#objective#`) while a Markov model or LLM fills
individual terminals with statistically/semantically richer content.

## 8. Resources

- **"Context-free grammar" — Wikipedia** — the formal definition (`N, Σ, R, S`), derivations, and
  the Chomsky hierarchy context: <https://en.wikipedia.org/wiki/Context-free_grammar>
- **Tracery (Kate Compton)** — the canonical lightweight grammar tool for generative text; play in
  the browser and read the model: <https://tracery.io/> and the docs at
  <http://www.crystalcodepalace.com/traceryTut.html>
- **"So you want to build a generator…" — Kate Compton** — essential practical essay on grammars
  and procedural content design: <https://www.galaxykate.com/apps/Prototypes/L-systems/>
- **PCG Book — "Grammars and L-systems" chapter** — procedural-generation framing of grammars:
  <http://pcgbook.com/>
- **Lark (Python parsing toolkit)** — when you need to *parse* with a CFG (EBNF, Earley/LALR):
  <https://lark-parser.readthedocs.io/>
- **NLTK `CFG` / generate** — build and sample grammars in Python for NLP experiments:
  <https://www.nltk.org/book/ch08.html>

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def derive(grammar, symbol, choose, max_depth=20, depth=0):
    """Expand a start symbol into terminals, bounded so recursion must terminate."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE